# NLP Disaster Tweets — BERTweet + 5-Fold CV

Улучшенный пайплайн поверх DistilBERT-бейзлайна (val F1 ≈ 0.8141).

**Ключевые изменения:**
1. **Модель** — `vinai/bertweet-base`: RoBERTa-архитектура, предобученная на 850M англоязычных твитов. Домен идеально совпадает с задачей.
2. **Конкатенация `keyword` + `location` + `text`** — раньше эти колонки игнорировались, хотя `keyword` сильно коррелирует с target.
3. **5-fold StratifiedKFold** — вместо одного train/val split. Снижает дисперсию оценки и использует все данные для обучения.
4. **Ensemble по fold'ам** — усреднение softmax-вероятностей с 5 моделей.
5. **Threshold tuning по OOF** — порог подбирается на out-of-fold предсказаниях, а не фиксируется 0.5.
6. **Overlap fix** — 68 текстов из train встречаются в test, для них берём готовые метки.

## 1. Импорты и настройки

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
    NUM_GPUS = 1
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    NUM_GPUS = torch.cuda.device_count()
else:
    DEVICE = torch.device('cpu')
    NUM_GPUS = 0

NUM_WORKERS = 4 if DEVICE.type == 'cuda' else 0
PIN_MEMORY  = DEVICE.type == 'cuda'

print(f'Device: {DEVICE}')
print(f'GPUs available: {NUM_GPUS}')
if DEVICE.type == 'cuda':
    for i in range(NUM_GPUS):
        props = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {props.name} ({props.total_memory // 1024**3} GB)')
print(f'PyTorch: {torch.__version__}')

## 2. Загрузка данных

In [ ]:
df_train = pd.read_csv('data/train.csv')
df_test  = pd.read_csv('data/test.csv')

print(f'Train: {df_train.shape},  Test: {df_test.shape}')
df_train.head()

## 3. Препроцессинг для BERTweet

BERTweet ожидает на вход твиты с нормализованными URL → `HTTPURL` и mentions → `@USER`.  
Эмодзи можно оставить — токенизатор знает unicode-символы.  
Хэштеги тоже оставляем как есть — у BERTweet есть отдельные токены под `#`-токены.

In [ ]:
def normalize_for_bertweet(text: str) -> str:
    text = re.sub(r'http\S+|www\S+', 'HTTPURL', text)
    text = re.sub(r'@\w+', '@USER', text)
    text = re.sub(r'&amp;', '&', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_train['clean_text'] = df_train['text'].apply(normalize_for_bertweet)
df_test['clean_text']  = df_test['text'].apply(normalize_for_bertweet)

for orig, clean in zip(df_train['text'][:3], df_train['clean_text'][:3]):
    print('ORIG :', orig)
    print('CLEAN:', clean)
    print()

## 4. Конкатенация keyword + location + text

`keyword` — это disaster-keyword, по которому Kaggle парсил твиты (`ablaze`, `flood`, `wreckage` и т.д.).  
Пропуски: 61 для keyword, 2533 для location — заменяем на пустую строку.  
Формат: `keyword | location | text`.

In [ ]:
def build_input(row) -> str:
    keyword  = str(row['keyword']).replace('%20', ' ') if pd.notna(row['keyword'])  else ''
    location = str(row['location'])                    if pd.notna(row['location']) else ''
    text     = row['clean_text']
    parts = [p for p in [keyword, location, text] if p]
    return ' | '.join(parts)

df_train['model_input'] = df_train.apply(build_input, axis=1)
df_test['model_input']  = df_test.apply(build_input,  axis=1)

for s in df_train['model_input'][:5]:
    print(s)
    print('-' * 80)

## 5. Токенизация и Dataset

In [ ]:
MODEL_NAME    = 'vinai/bertweet-base'
MAX_LEN       = 128
EPOCHS        = 3
LEARNING_RATE = 2e-5
N_SPLITS      = 5

BATCH_SIZE_PER_DEVICE = 32
BATCH_SIZE = BATCH_SIZE_PER_DEVICE * max(NUM_GPUS, 1)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False, normalization=False)
print(f'Tokenizer loaded | batch size: {BATCH_SIZE} ({BATCH_SIZE_PER_DEVICE} × {max(NUM_GPUS, 1)} GPU(s))')

In [ ]:
class TweetDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            padding='max_length',
            max_length=MAX_LEN,
            return_tensors='pt'
        )
        self.labels = labels.reset_index(drop=True) if labels is not None else None

    def __len__(self):
        return self.encodings['input_ids'].shape[0]

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels.iloc[idx], dtype=torch.long)
        return item

## 6. Функции train/eval

`eval_epoch` возвращает не argmax, а вероятности класса 1 — они нужны для усреднения по fold'ам и подбора порога.

In [ ]:
from tqdm.auto import tqdm

def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0
    base = model.module if isinstance(model, nn.DataParallel) else model

    for batch in tqdm(loader, desc='Train', leave=False):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        output = model(**batch)
        loss = output.loss.mean()

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(base.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    return total_loss / len(loader)


def predict_probs(model, loader, has_labels=True):
    model.eval()
    all_probs, all_labels = [], []

    with torch.no_grad():
        for batch in tqdm(loader, desc='Predict', leave=False):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            output = model(**batch)
            probs = torch.softmax(output.logits, dim=1)[:, 1].cpu().numpy()
            all_probs.extend(probs)
            if has_labels:
                all_labels.extend(batch['labels'].cpu().numpy())

    return np.array(all_probs), (np.array(all_labels) if has_labels else None)

## 7. 5-Fold StratifiedKFold обучение

Для каждого fold'а:
1. Создаём свежую модель и оптимизатор
2. Обучаем 3 эпохи
3. Сохраняем OOF-вероятности для val-части
4. Сохраняем test-вероятности (один прогон test'а через эту модель)

Итог: `oof_probs[i]` — усреднённая вероятность класса 1 для i-го train-примера (получена моделью, не видевшей его на обучении). `test_probs` — средние вероятности по 5 fold'ам.

In [ ]:
X = df_train['model_input'].values
y = df_train['target'].values
X_test_texts = df_test['model_input'].values

test_dataset = TweetDataset(pd.Series(X_test_texts))
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

oof_probs  = np.zeros(len(df_train))
test_probs = np.zeros(len(df_test))
fold_scores = []

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f'\n{"="*60}\nFold {fold}/{N_SPLITS}\n{"="*60}')

    train_ds = TweetDataset(pd.Series(X[train_idx]), pd.Series(y[train_idx]))
    val_ds   = TweetDataset(pd.Series(X[val_idx]),   pd.Series(y[val_idx]))

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    model = model.to(DEVICE)
    if NUM_GPUS > 1:
        model = nn.DataParallel(model)
    base_model = model.module if isinstance(model, nn.DataParallel) else model

    optimizer = torch.optim.AdamW(base_model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
    total_steps  = EPOCHS * len(train_loader)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=total_steps // 10, num_training_steps=total_steps
    )

    val_p, val_y = None, None
    for epoch in range(1, EPOCHS + 1):
        train_loss = train_epoch(model, train_loader, optimizer, scheduler)
        val_p, val_y = predict_probs(model, val_loader)
        val_f1 = f1_score(val_y, (val_p > 0.5).astype(int))
        print(f'  Epoch {epoch}: train_loss={train_loss:.4f}  val_F1@0.5={val_f1:.4f}')

    oof_probs[val_idx] = val_p
    fold_f1 = f1_score(val_y, (val_p > 0.5).astype(int))
    fold_scores.append(fold_f1)
    print(f'  Fold {fold} OOF F1@0.5: {fold_f1:.4f}')

    test_p, _ = predict_probs(model, test_loader, has_labels=False)
    test_probs += test_p / N_SPLITS

    del model, base_model, optimizer, scheduler
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

print(f'\n{"="*60}')
print(f'Mean fold F1@0.5: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}')
print(f'Overall OOF F1@0.5: {f1_score(y, (oof_probs > 0.5).astype(int)):.4f}')

## 8. Threshold tuning по OOF

F1 чувствителен к порогу. Подбираем оптимальный по out-of-fold вероятностям, потом применим к test.

In [ ]:
thresholds = np.arange(0.30, 0.61, 0.01)
f1_per_threshold = [f1_score(y, (oof_probs > t).astype(int)) for t in thresholds]

best_idx = int(np.argmax(f1_per_threshold))
best_threshold = thresholds[best_idx]
best_oof_f1 = f1_per_threshold[best_idx]

print(f'Best threshold: {best_threshold:.2f}  →  OOF F1 = {best_oof_f1:.4f}')
print(f'(F1@0.5 для сравнения: {f1_score(y, (oof_probs > 0.5).astype(int)):.4f})')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thresholds, f1_per_threshold, marker='o', markersize=3)
ax.axvline(best_threshold, color='red', linestyle='--', label=f'best={best_threshold:.2f}')
ax.set_xlabel('Threshold')
ax.set_ylabel('OOF F1')
ax.set_title('F1 vs threshold (по OOF предсказаниям)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Отчёт и confusion matrix на OOF

In [ ]:
oof_preds = (oof_probs > best_threshold).astype(int)
print(classification_report(y, oof_preds, target_names=['Not Disaster', 'Disaster']))

cm = confusion_matrix(y, oof_preds)
ConfusionMatrixDisplay(cm, display_labels=['Not Disaster', 'Disaster']).plot(cmap='Blues')
plt.title(f'OOF Confusion Matrix (F1={best_oof_f1:.4f}, threshold={best_threshold:.2f})')
plt.show()

## 10. Test предсказания и overlap fix

68 текстов из train встречаются в test — для них берём метку из train напрямую, минуя модель.

In [ ]:
test_preds = (test_probs > best_threshold).astype(int).tolist()
print(f'До overlap fix → Disaster: {sum(test_preds)}, Not disaster: {len(test_preds) - sum(test_preds)}')

def majority_label(labels):
    counts = labels.value_counts()
    if len(counts) > 1 and counts.iloc[0] == counts.iloc[1]:
        return 1
    return counts.idxmax()

train_dedup = df_train.groupby('text', sort=False)['target'].agg(majority_label)
overlap_labels = train_dedup[train_dedup.index.isin(df_test['text'])].to_dict()

overridden = 0
for i, text in enumerate(df_test['text']):
    if text in overlap_labels:
        test_preds[i] = overlap_labels[text]
        overridden += 1

print(f'Overlap fix: перезаписано {overridden} предсказаний ({len(overlap_labels)} уникальных текстов)')
print(f'После overlap fix → Disaster: {sum(test_preds)}, Not disaster: {len(test_preds) - sum(test_preds)}')

## 11. Submission

In [ ]:
submission = pd.read_csv('data/sample_submission.csv')
submission['target'] = test_preds
submission.to_csv('submission_bertweet_kfold.csv', index=False)

print('submission_bertweet_kfold.csv saved')
submission.head()